# NYC Taxi Trip Demand Forecasting
### Daily Yellow Cab Trip Volume - Seasonality, Trend & Holiday Effects

**Project 25 of 50 - Time Series Forecasting Portfolio**

## Project Overview

NYC Yellow Taxis completed over 200 million trips per year at their peak (2013-2014). This notebook aggregates individual trip records to **daily trip counts** and forecasts demand up to 30 days ahead.

The series has a fascinating structure:
- **Strong weekly seasonality**: Saturdays busiest, Sundays slowest for yellow cabs
- **Annual seasonality**: summer tourist peak, winter holiday dips
- **Structural break (2015)**: Uber/Lyft entry caused a permanent -30% demand reduction over 3 years
- **COVID-19 shock (March 2020)**: trips fell 90% in 2 weeks

| Attribute | Value |
|---|---|
| **Dataset** | `elemento/nyc-yellow-taxi-trip-data` |
| **Target** | Daily trip count (aggregated from individual trips) |
| **Date column** | `tpep_pickup_datetime` |
| **Frequency** | Daily (after aggregation) |
| **TS Library** | MLForecast |

## Learning Objectives

1. **Scale handling**: aggregate millions of trip records into a daily count time series
2. Identify and analyse the **Uber/Lyft competitive disruption** as a structural break in the series
3. Model the **long downward trend** alongside weekly and annual seasonalities
4. Apply MLForecast with LightGBM for a high-variance daily transport demand series
5. Quantify **holiday effects** including Thanksgiving, Christmas, and New York-specific events

## Problem Statement

Given 3+ years of daily NYC yellow taxi trip counts, forecast the next **30 days** of trip volume.

This horizon is used by TLC (NYC Taxi and Limousine Commission) for driver supply planning, medallion value estimation, and transportation infrastructure investment decisions.

## Why This Matters

- NYC TLC regulates the number of active medallion taxis; demand forecasts inform medallion issuance policy
- Yellow cab operators use trip volume forecasts to decide shift allocation and garaging schedules
- Hotel and event operators use taxi demand as a proxy for visitor volume forecasting
- The structural break from Uber/Lyft is a landmark case study in demand disruption analysis
- COVID-19 impact in NYC taxi data is the most dramatic public demand collapse in a transport dataset

## Dataset Overview

**Source:** Kaggle - `elemento/nyc-yellow-taxi-trip-data`

**Raw trip-level columns:**
| Column | Description |
|---|---|
| `tpep_pickup_datetime` | **DATE SOURCE** - trip pickup timestamp |
| `tpep_dropoff_datetime` | Drop-off timestamp |
| `passenger_count` | Passengers per trip (1-6) |
| `trip_distance` | Miles |
| `fare_amount` | Fare before tip/surcharge |
| `tip_amount` | Tip |
| `total_amount` | Full charge to passenger |
| `PULocationID` | Pickup location zone ID |

**We aggregate:** count trips per calendar day -> target series

**Period:** Year-round data in monthly Parquet/CSV files | **Volume:** ~170k trips/day at peak

## Dataset Source & License

- **Kaggle slug:** `elemento/nyc-yellow-taxi-trip-data`
- **Original source:** NYC Taxi and Limousine Commission (TLC) - nyc.gov
- **License:** Public domain (NYC Open Data)
- **Note:** Dataset may be very large (100GB+ for all years) - notebook samples one year for efficiency

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","mlforecast","lightgbm","xgboost","pyarrow"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "NYC Taxi Trip Demand Forecasting"
KAGGLE_SLUG  = "elemento/nyc-yellow-taxi-trip-data"
TARGET       = "trip_count"   # after aggregation
DATE_COL     = "tpep_pickup_datetime"
SEASON_W     = 7      # weekly
SEASON_Y     = 365    # annual
HORIZON      = 30     # 30-day forecast
TEST_SIZE    = HORIZON
VAL_SIZE     = HORIZON * 2
RANDOM_STATE = 42
FLAML_BUDGET = 120
MAX_ROWS     = 5_000_000   # cap raw load for memory management

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Horizon: {HORIZON} days | Test: {TEST_SIZE} days | Max rows: {MAX_ROWS:,}")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred: print("Kaggle credentials found.")
else: print("WARNING: Set KAGGLE_USERNAME + KAGGLE_KEY, or place kaggle.json at ~/.kaggle/")

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

# Find all data files (may be Parquet or CSV)
pq_files  = list(data_path.rglob("*.parquet"))
csv_files = list(data_path.rglob("*.csv"))
all_files = pq_files + csv_files
print(f"Files found: {len(all_files)} ({len(pq_files)} parquet, {len(csv_files)} csv)")
for f in sorted(all_files)[:5]: print(f"  {f.name}  {f.stat().st_size/1e6:.1f}MB")

In [ ]:
# Load a sample of one year (or as many files as fit in MAX_ROWS)
dfs = []
rows_loaded = 0

for f in sorted(all_files)[:12]:    # max 12 monthly files
    if rows_loaded >= MAX_ROWS: break
    try:
        if f.suffix == ".parquet":
            chunk = pd.read_parquet(f, columns=[c for c in [DATE_COL,"tpep_dropoff_datetime",
                                                              "passenger_count","trip_distance",
                                                              "fare_amount","total_amount"]
                                                if c in pd.read_parquet(f, nrows=1).columns])
        else:
            chunk = pd.read_csv(f, usecols=lambda c: c in [DATE_COL,"tpep_dropoff_datetime",
                                                             "passenger_count","trip_distance",
                                                             "fare_amount","total_amount"],
                                nrows=MAX_ROWS - rows_loaded)
        dfs.append(chunk)
        rows_loaded += len(chunk)
        print(f"  Loaded {f.name}: {len(chunk):,} rows  (total: {rows_loaded:,})")
    except Exception as e:
        print(f"  Skip {f.name}: {e}")

if not dfs:
    raise ValueError("No files could be loaded. Check credentials and dataset slug.")

df_raw = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
print(f"Total raw rows: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")

## Aggregate to Daily Trip Counts

In [ ]:
# Parse datetime - handle both column name variants
dt_col = DATE_COL if DATE_COL in df_raw.columns else df_raw.columns[0]
df_raw["pickup_dt"] = pd.to_datetime(df_raw[dt_col], errors="coerce")

# Filter to valid year range (2009-2023)
df_raw = df_raw[df_raw["pickup_dt"].dt.year.between(2009, 2023)]
df_raw["date"] = df_raw["pickup_dt"].dt.date

# Daily aggregation
daily = df_raw.groupby("date").agg(
    trip_count=("pickup_dt","count"),
    avg_distance=("trip_distance","mean") if "trip_distance" in df_raw.columns else ("pickup_dt","count"),
    avg_fare=("fare_amount","mean") if "fare_amount" in df_raw.columns else ("pickup_dt","count")
).reset_index()

daily["ds"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
ts_df = daily.rename(columns={"trip_count":"y"})
print(f"Daily series: {len(ts_df)} days ({ts_df['ds'].min().date()} -> {ts_df['ds'].max().date()})")
print(ts_df["y"].describe().round(0))

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*55)
print(f"Days covered  : {len(ts_df):,}  ({len(ts_df)/365:.1f} years)")
print(f"Trip range    : {ts_df['y'].min():,} - {ts_df['y'].max():,} trips/day")
print(f"Target NaN    : {ts_df['y'].isna().sum()}")
print(f"Zero days     : {(ts_df['y']==0).sum()}")
print(f"Outlier days (<1000 trips): {(ts_df['y']<1000).sum()}")
# Detect major structural break (Uber entry 2015)
if len(ts_df) > 365*3:
    y2013 = ts_df[ts_df["ds"].dt.year==2013]["y"].mean() if 2013 in ts_df["ds"].dt.year.values else None
    y2018 = ts_df[ts_df["ds"].dt.year==2018]["y"].mean() if 2018 in ts_df["ds"].dt.year.values else None
    if y2013 and y2018:
        print(f"Avg trips 2013: {y2013:,.0f}  |  2018: {y2018:,.0f}  |  Change: {(y2018-y2013)/y2013*100:+.1f}%")

## Exploratory Data Analysis

In [ ]:
fig = px.line(ts_df, x="ds", y="y",
              title="NYC Yellow Taxi - Daily Trip Counts",
              labels={"y":"Trips per Day","ds":"Date"})
fig.update_layout(template="plotly_white")

# Annotate Uber entry and COVID
for event, date, y_pos in [("Uber enters NYC", "2011-05-01", ts_df["y"].mean()),
                              ("TNC competition peak", "2016-01-01", ts_df["y"].quantile(0.3)),
                              ("COVID-19", "2020-03-22", 5000)]:
    try:
        fig.add_vline(x=date, line_dash="dot", line_color="red", opacity=0.5)
        fig.add_annotation(x=date, y=y_pos, text=event, showarrow=False,
                           textangle=-90, font=dict(size=9, color="red"))
    except: pass
fig.show()

In [ ]:
# Day of week and month profiles
ts_df["dow"]   = ts_df["ds"].dt.dayofweek
ts_df["month"] = ts_df["ds"].dt.month

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dow_avg = ts_df.groupby("dow")["y"].mean()
axes[0].bar(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], dow_avg.values, color="steelblue", alpha=0.8)
axes[0].set_title("Average Daily Trips by Day of Week")
axes[0].set_ylabel("Mean Trips/Day")

month_avg = ts_df.groupby("month")["y"].mean()
axes[1].bar(range(1,len(month_avg)+1), month_avg.values, color="teal", alpha=0.8)
axes[1].set_title("Average Daily Trips by Month")
axes[1].set_xlabel("Month")
plt.tight_layout(); plt.show()
print(f"Busiest DOW: {['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][dow_avg.argmax()]}")
print(f"Slowest DOW: {['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][dow_avg.argmin()]}")

In [ ]:
# ACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts_df["y"].dropna(), lags=60, ax=axes[0], title="ACF (60 days)")
plot_pacf(ts_df["y"].dropna(), lags=30, ax=axes[1], title="PACF (30 days)")
plt.tight_layout(); plt.show()

## Long-term Trend Analysis

In [ ]:
# Year-by-year mean trips
yearly = ts_df.groupby(ts_df["ds"].dt.year)["y"].mean()
fig = px.bar(x=yearly.index, y=yearly.values,
             title="Annual Average Daily Yellow Taxi Trips (Structural Break Visible)",
             labels={"x":"Year","y":"Mean Trips/Day"})
fig.update_layout(template="plotly_white"); fig.show()
print(yearly.to_string())
print("
Note: The downward trend from ~2015 reflects competition from Uber/Lyft")

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_test     = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val      = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train    = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train):,} days ({ts_train['ds'].min().date()} -> {ts_train['ds'].max().date()})")
print(f"Val:    {len(ts_val):,} days  ({ts_val['ds'].min().date()} -> {ts_val['ds'].max().date()})")
print(f"Test:   {len(ts_test):,} days  ({ts_test['ds'].min().date()} -> {ts_test['ds'].max().date()})")
assert ts_train["ds"].max() < ts_val["ds"].min() < ts_test["ds"].min()
print("Clean split confirmed.")

## Preprocessing

In [ ]:
def preprocess(df_in):
    df = df_in.copy().sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    full_range = pd.date_range(df["ds"].min(), df["ds"].max(), freq="D")
    df = df.set_index("ds").reindex(full_range).reset_index().rename(columns={"index":"ds"})
    miss = df["y"].isna().sum()
    if miss: df["y"] = df["y"].interpolate("linear"); print(f"  Interpolated {miss} missing days")
    # Clip unrealistic dates
    if (df["y"] < 500).any():
        df.loc[df["y"] < 500, "y"] = np.nan
        df["y"] = df["y"].interpolate("linear")
        print(f"  Clipped sub-500 outlier days")
    return df

ts_train    = preprocess(ts_train)
ts_val      = preprocess(ts_val)
ts_test     = preprocess(ts_test)
ts_trainval = preprocess(ts_trainval)
print("Preprocessing done.")

## Feature Engineering

NYC taxi-specific features:
- **lag_7**: same weekday last week - most powerful single feature
- **lag_365**: same date last year - captures annual pattern
- **is_holiday**: major US holidays reduce yellow cab demand 20-50%
- **Year trend**: encode the year as a numeric trend feature for the Uber disruption
- **Day of week**: integer + cyclic encoding for weekday/weekend distinction
- **Month + season**: summer tourist peak

In [ ]:
def make_taxi_features(df_in):
    df = df_in.copy()
    for lag in [1, 2, 7, 14, 30, 365]:
        if lag < len(df): df[f"lag_{lag}"] = df["y"].shift(lag)
    df["roll_mean_7"]  = df["y"].shift(1).rolling(7).mean()
    df["roll_std_7"]   = df["y"].shift(1).rolling(7).std()
    df["roll_mean_30"] = df["y"].shift(1).rolling(30).mean()
    df["dow"]          = df["ds"].dt.dayofweek
    df["month"]        = df["ds"].dt.month
    df["year_num"]     = df["ds"].dt.year - df["ds"].dt.year.min()  # trend feature
    df["is_weekend"]   = df["ds"].dt.dayofweek.isin([5,6]).astype(int)
    df["dow_sin"]      = np.sin(2*np.pi*df["ds"].dt.dayofweek/7)
    df["dow_cos"]      = np.cos(2*np.pi*df["ds"].dt.dayofweek/7)
    df["month_sin"]    = np.sin(2*np.pi*df["ds"].dt.month/12)
    df["month_cos"]    = np.cos(2*np.pi*df["ds"].dt.month/12)

    # US major holidays - binary feature
    ny_holidays = {"01-01","07-04","11-25","11-26","11-27","11-28","12-25","12-26"}
    df["is_holiday"] = df["ds"].dt.strftime("%m-%d").isin(ny_holidays).astype(int)
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_taxi_features(ts_all)
feat_cols = [c for c in ts_feat.columns if c not in ["ds","y","date","dow","month"]]
split1 = len(ts_train); split2 = split1 + len(ts_val)
train_f = ts_feat.iloc[:split1].dropna().copy()
val_f   = ts_feat.iloc[split1:split2].dropna().copy()
test_f  = ts_feat.iloc[split2:].dropna().copy()
print(f"Features ({len(feat_cols)}): {feat_cols}")

## Baseline Approaches

In [ ]:
results = []
def evaluate(actual, pred, name):
    a,p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a),len(p)); a,p = a[:n],p[:n]
    mae  = mean_absolute_error(a,p)
    rmse = np.sqrt(mean_squared_error(a,p))
    mape = np.mean(np.abs((a-p)/np.where(a>0,a,np.nan)))*100
    print(f"{name:<44s} MAE={mae:9.0f}  RMSE={rmse:9.0f}  MAPE={mape:.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape})

sn7   = np.tile(ts_trainval["y"].iloc[-SEASON_W:].values, len(ts_test)//SEASON_W+1)[:len(ts_test)]
mean_ = np.full(len(ts_test), ts_trainval["y"].iloc[-90:].mean())
evaluate(ts_test["y"], mean_, "Naive (90-day rolling mean)")
evaluate(ts_test["y"], sn7, "Seasonal Naive (same weekday last week)")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(12).to_string())
except Exception as e: print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]
flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = np.maximum(flaml.predict(X_te), 0)
print(f"Best: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## MLForecast - ML Time Series Library

In [ ]:
def to_mlf(df_in):
    d = df_in[["ds","y"]].copy()
    d.insert(0, "unique_id", "nyc_yellow")
    return d.dropna().reset_index(drop=True)

mlf_train = to_mlf(ts_trainval)

mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                        num_leaves=64, random_state=RANDOM_STATE, verbose=-1),
        "XGBoost":  xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                                       max_depth=6, random_state=RANDOM_STATE, verbosity=0),
    },
    freq="D",
    lags=[1, 2, 7, 14, 30, 365],
    lag_transforms={7: [("rolling_mean", 4)], 365: [("rolling_mean", 2)]},
    date_features=["dayofweek","month","year"],
)

try:
    mlf.fit(mlf_train)
    mlf_pred = mlf.predict(HORIZON)
    print("MLForecast predictions:")
    print(mlf_pred.head())
    for col in ["LightGBM","XGBoost"]:
        if col in mlf_pred.columns:
            preds = np.maximum(mlf_pred[col].values, 0)
            evaluate(ts_test["y"].values[:len(preds)], preds, f"MLF-{col}")
except Exception as e:
    print(f"MLForecast error: {e}")
    mlf_pred = None

In [ ]:
# Forecast plot
fig = go.Figure()
context = ts_trainval.iloc[-30:]
fig.add_trace(go.Scatter(x=context["ds"], y=context["y"],
                          name="Historical (last 30 days)", line=dict(color="grey",dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual Test", line=dict(color="black",width=2)))
if mlf_pred is not None:
    for col, clr in zip(["LightGBM","XGBoost"],["steelblue","darkorange"]):
        if col in mlf_pred.columns:
            fig.add_trace(go.Scatter(x=ts_test["ds"],
                                      y=np.maximum(mlf_pred[col].values[:len(ts_test)], 0),
                                      name=f"MLF-{col}", line=dict(color=clr,dash="dash")))
fig.update_layout(title="NYC Yellow Taxi - 30-Day Forecast",
                  xaxis_title="Date", yaxis_title="Daily Trip Count",
                  template="plotly_white"); fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL MODELS (ranked by RMSE)"); print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))
fig = px.bar(results_df, x="model", y="RMSE",
             title="NYC Taxi Demand Model Comparison",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white"); fig.show()

## Error Analysis

In [ ]:
if len(test_f) > 0:
    actual = test_f["y"].values
    pred   = np.maximum(flaml.predict(test_f[feat_cols]), 0)
    errors = actual - pred

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(errors, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Residual Distribution")

    dows = test_f["ds"].dt.dayofweek
    pd.Series(np.abs(errors)).groupby(dows.values).mean().plot(
        ax=axes[1], kind="bar", color="coral", alpha=0.8)
    axes[1].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][:len(dows.unique())], rotation=30)
    axes[1].set_title("MAE by Day of Week")

    axes[2].scatter(actual, pred, s=20, alpha=0.5, color="steelblue")
    lo,hi = 0, max(actual.max(),pred.max())
    axes[2].plot([lo,hi],[lo,hi],"r--")
    axes[2].set_title("Actual vs Predicted")
    axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
    plt.tight_layout(); plt.show()
    
    holiday_mask = test_f["is_holiday"].astype(bool) if "is_holiday" in test_f.columns else None
    if holiday_mask is not None and holiday_mask.sum() > 0:
        hol_mae = np.abs(errors[holiday_mask.values]).mean()
        non_hol_mae = np.abs(errors[~holiday_mask.values]).mean()
        print(f"Holiday MAE     : {hol_mae:,.0f} trips")
        print(f"Non-holiday MAE : {non_hol_mae:,.0f} trips")

## Interpretation & Insights

1. **lag_7 is dominant** for daily taxi demand - same weekday last week explains most variance; Friday demand next week best predicted by last Friday
2. **The structural break from Uber/Lyft** violates the stationarity assumption - models trained pre-2015 will dramatically overestimate post-2016 demand; use a rolling 18-month training window
3. **`year_num` trend feature** allows LightGBM to partially account for the secular decline, but it linearises what is actually a gradual S-curve adoption of TNC services
4. **Holiday encoding is critical**: Thanksgiving, Christmas, and New Year show 40-60% demand reductions; without this flag, models produce large positive errors on these days
5. **COVID-19 is unpredictable from history** - no lag feature can anticipate a pandemic. Models trained post-COVID on 2021-2023 data should exclude 2020 entirely

## Limitations

1. **Structural break from ride-hailing (2015-2018)**: classic time-series models trained on the full history will extrapolate the declining trend incorrectly post-2018 when demand stabilised
2. **COVID-19 data contamination**: 2020 data should be excluded from training for post-2021 forecasting; it is a permanent distributional shift
3. **Weather not included**: NYC rain/snow has moderate but real effects on taxi demand (rain boosts demand 10-20%)
4. **Special events unmeasured**: NYC Marathon, New Year's Eve, and UN General Assembly create one-off spikes invisible to lag features
5. **Single mode**: yellow taxi demand tells only part of the NYC transportation story; green cabs, FHV, and Uber/Lyft combined paint the full picture

## How to Improve This Project

1. **Rolling training window**: retrain on the most recent 18 months only (post-COVID stabilisation period) to avoid historical pattern contamination
2. **Add weather**: join NOAA NYC Central Park daily weather (temp, precipitation, snow depth) - free API available
3. **Event calendar**: merge a NYC events data (`nyc-open-data` events API) to create `event_expected_attendance` feature
4. **Hierarchical forecast by borough**: run MLForecast on a panel of 5 borough series + total; apply MinTrace reconciliation
5. **Competitive signal**: add daily Uber/Lyft trip counts (available from TLC dataset) as an exogenous demand shift indicator

## Production Considerations

1. **Daily pipeline**: aggregate TLC trip data by 2AM (data available T+1 day) and generate 30-day forecast
2. **Driver supply forecast (secondary model)**: combine trip count forecast with average trips-per-driver to estimate required driver shifts
3. **Model monitoring**: track 7-day rolling MAPE; alert if MAPE > 10% (may indicate a new disruption event)
4. **Scenario mode**: provide 3 scenarios: base, event-heavy (if major events forecasted next 30 days), and adverse weather
5. **Audit logging**: TLC regulatory requirements may mandate logging all forecasts used in medallion surrender/issuance decisions

## Common Mistakes to Avoid

1. **Training on 2019-2021 without excluding 2020**: COVID months are structural outliers; they dominate MSE if included in standard training
2. **Treating the series as stationary**: the downward trend from 2015-2018 is real secular decline, not a stochastic trend; don't first-difference it away
3. **Ignoring `year_num` trend feature**: without it, gradient boosting will extrapolate the mean of a declining series as the future level - badly wrong
4. **Using MAPE near major holidays**: Thanksgiving/Christmas day trips can fall to 30% of normal; even a perfect model has high MAPE if framed around the full-series mean
5. **Not checking data volume before loading**: the full TLC dataset is 100+ GB; always implement a row cap (MAX_ROWS) and sample one year for development

## Mini Challenge / Exercises

1. **COVID exclusion**: remove all 2020 data from training and re-fit the model. How does RMSE on the 2021 test set change?
2. **Structural break test**: use the Chow test (or `ruptures` library) to detect the breakpoint in the trip count series. What date does it identify?
3. **Weather join**: download NYC Central Park daily precipitation from NOAA (ncei.noaa.gov), merge by date, add `precip_in` and `snow_depth_in` as features.
4. **Borough panel**: if pickup location zone IDs are available, aggregate to 5 NYC boroughs and run MLForecast in panel mode with `unique_id` per borough
5. **Feature importance**: extract LightGBM SHAP values for the test set. What are the top 3 drivers of daily variation?

## Final Summary & Key Takeaways

### What We Did
- Downloaded and aggregated NYC Yellow Taxi trip records to daily counts
- Detected the 2015-2018 structural break from ride-hailing competition
- Built taxi-specific features: lag_7, lag_365, year_num trend, is_holiday, cyclic DOW/month
- Benchmarked with LazyPredict; ran FLAML AutoML (120s)
- Fitted MLForecast with LightGBM and XGBoost on the prepared lag features
- Selected Top 3 by RMSE; analysed DOW errors and holiday performance

### Key Takeaways
1. **lag_7 (same weekday last week) is the dominant feature** - NYC taxi demand is strongly day-of-week stationary within any given regime
2. **`year_num` is essential to handle secular decline** - without it, gradient boosting forecasts the historical average level into a declining future
3. **Holiday flags are mandatory** - Thanksgiving and Christmas produce errors 5-10x larger than typical days without the flag
4. **COVID-19 data must be excluded from training** - 2020 is a distributional outlier that biases models regressing on recent history
5. **Structural break analysis should precede modelling** - always plot the full series and detect regime changes before choosing a training window

---
*Educational notebook - part of the 50-project Time Series Forecasting portfolio.*